# 🌐 01. 환경설정과 첫 크롤링
## — VS Code + Python으로 시작하기

> **이 노트북에서 배울 것**
> 1. 왜 VS Code를 쓰는가 (Colab 대신)
> 2. 필요한 패키지 설치
> 3. `requests`로 웹페이지 가져오기
> 4. HTTP 상태 코드 이해
> 5. User-Agent 헤더 다루기
> 6. robots.txt 확인하는 습관


## 1. 왜 VS Code인가? (Colab과의 차이)

```
구글 Colab의 문제점:
┌─────────────────────────────────────────┐
│  클라우드 서버 IP  ──→  사이트가 차단!  │
│  올리브영, 무신사 등 대부분의 쇼핑몰    │
│  → 403 Forbidden (접근 거부)            │
└─────────────────────────────────────────┘

VS Code (내 컴퓨터)의 장점:
┌─────────────────────────────────────────┐
│  내 집 IP  ──→  일반 사용자처럼 접속!  │
│  → 차단될 확률 훨씬 낮음               │
└─────────────────────────────────────────┘
```

> 💡 **핵심**: 크롤링은 항상 자신의 컴퓨터에서 실행하세요.


## 2. 패키지 설치

VS Code의 터미널(`Ctrl + 백틱`)을 열고 아래 명령어를 실행하세요.

```bash
pip install requests beautifulsoup4 pandas
```

설치 후 아래 셀을 실행해서 확인합니다.


In [1]:
# 패키지 설치 확인
# ─ 에러 없이 실행되면 설치 성공!

import requests
import bs4          # beautifulsoup4
import pandas as pd

print(f"requests 버전: {requests.__version__}")
print(f"BeautifulSoup 버전: {bs4.__version__}")
print(f"pandas 버전: {pd.__version__}")
print("\n✅ 모든 패키지 설치 완료!")


requests 버전: 2.34.0
BeautifulSoup 버전: 4.14.3
pandas 버전: 3.0.3

✅ 모든 패키지 설치 완료!


## 3. HTTP가 뭔가요?

웹 크롤링의 원리를 이해하려면 HTTP를 알아야 해요.

```
[내 컴퓨터]  ─── 요청(Request) ──→  [웹 서버]
             ←── 응답(Response) ───

요청: "이 URL의 페이지를 보여줘!"
응답: "여기 HTML 파일이야!"
```

`requests` 라이브러리는 이 과정을 파이썬 코드로 자동으로 해줍니다.


In [2]:
# 첫 번째 크롤링!
# ─ 연습용 사이트 books.toscrape.com

import requests

url = 'http://books.toscrape.com/'
response = requests.get(url)

print(f"상태 코드: {response.status_code}")
print(f"응답 크기: {len(response.text):,} 글자")
print(f"\n--- HTML 앞부분 미리보기 ---")
print(response.text[:300])


상태 코드: 200
응답 크기: 51,294 글자

--- HTML 앞부분 미리보기 ---
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lan


## 4. 상태 코드 — 숫자로 읽는 서버의 대답

| 코드 | 의미 | 우리가 해야 할 것 |
|------|------|-----------------|
| **200** | 성공! ✅ | 바로 진행 |
| **403** | 접근 거부 ❌ | User-Agent 헤더 추가 |
| **404** | 페이지 없음 | URL 재확인 |
| **429** | 요청 너무 많음 | `time.sleep()` 추가 |
| **500** | 서버 에러 | 잠시 후 재시도 |

> 💡 크롤링에서 가장 많이 만나는 것: **200** (성공)과 **403** (차단)


In [3]:
# 상태 코드 확인 패턴
import requests

def check_status(url):
    response = requests.get(url, timeout=10)

    if response.status_code == 200:
        print(f"✅ 성공! ({url})")
    elif response.status_code == 403:
        print(f"❌ 차단됨! User-Agent를 추가해보세요. ({url})")
    elif response.status_code == 404:
        print(f"⚠️  페이지 없음! URL을 확인하세요. ({url})")
    else:
        print(f"상태 코드: {response.status_code} ({url})")

    return response.status_code

check_status('http://books.toscrape.com/')
check_status('http://books.toscrape.com/없는페이지')


✅ 성공! (http://books.toscrape.com/)
⚠️  페이지 없음! URL을 확인하세요. (http://books.toscrape.com/없는페이지)


404

## 5. User-Agent — 내가 누구인지 서버에 알려주기

서버는 요청을 보낸 게 **사람인지 봇인지** 확인해요.
아무 헤더 없이 보내면 파이썬 봇으로 인식해서 차단할 수 있어요.

```
헤더 없을 때:   User-Agent: python-requests/2.31.0  ← 봇처럼 보임
헤더 추가 후:   User-Agent: Mozilla/5.0 ... Chrome/120  ← 크롬 브라우저처럼 보임
```

> 📋 User-Agent 복사하는 법:
> 1. 크롬 브라우저 열기
> 2. F12 → Console 탭
> 3. `navigator.userAgent` 입력 후 Enter
> 4. 나온 문자열을 복사!


In [6]:
import requests

imhuman = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36'
}
response = requests.get('http://books.toscrape.com/')
print(f"헤더 없음: {response.status_code}")

헤더 없음: 200


In [4]:
# User-Agent 헤더 추가
import requests

# ❌ 헤더 없이 요청 — 일부 사이트에서 차단될 수 있음
response_no_header = requests.get('http://books.toscrape.com/')
print(f"헤더 없음: {response_no_header.status_code}")

# ✅ User-Agent 추가 — 일반 크롬 브라우저처럼 보임
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
response_with_header = requests.get('http://books.toscrape.com/', headers=headers)
print(f"헤더 있음: {response_with_header.status_code}")

# 앞으로 항상 이 headers를 기본으로 사용합니다!


헤더 없음: 200
헤더 있음: 200


## 6. 인코딩 — 한글 깨짐 방지

한국 사이트에서 한글이 깨지는 경우가 있어요.
원인: 서버가 `EUC-KR`로 보냈는데 파이썬이 다른 방식으로 읽는 것.

```python
response.encoding = 'utf-8'     # 대부분의 현대 사이트
response.encoding = 'euc-kr'    # 오래된 한국 사이트
```


In [5]:
# 인코딩 처리 비교
import requests

url = 'https://www.saramin.co.kr/zf_user/search/recruit?searchword=파이썬'
headers = {'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36'}

response = requests.get(url, headers=headers, timeout=15)

# 자동 감지된 인코딩 확인
print(f"자동 인코딩: {response.encoding}")

# 명시적으로 utf-8 설정 (한글 깨짐 방지)
response.encoding = 'utf-8'

# 한글이 포함된 텍스트 일부 확인
import re
korean_texts = re.findall('[가-힣]+', response.text[:2000])
print(f"\n발견된 한글: {korean_texts[:10]}")


자동 인코딩: UTF-8

발견된 한글: ['파이썬', '채용정보', '총', '건의', '검색결과', '사람인', '파이썬', '취업', '구인', '구직']


## 7. robots.txt — 크롤링 전 필수 확인!

모든 사이트에는 `robots.txt` 파일이 있어요.
이것은 "어디까지 크롤링을 허용하는가"를 명시한 규칙서입니다.

```
https://사이트주소.com/robots.txt
```

```
User-agent: *          ← 모든 봇에 적용
Disallow: /private/    ← 이 경로는 수집 금지!
Disallow: /mypage/     ← 이 경로도 금지!
Allow: /products/      ← 이 경로는 허용 ✅
```

> ⚠️ **Disallow된 경로를 크롤링하면 법적 문제가 될 수 있어요!**


In [8]:
# robots.txt 확인 함수
import requests

def check_robots(site_url):
    """사이트의 robots.txt를 확인하고 출력합니다"""
    robots_url = site_url.rstrip('/') + '/robots.txt'

    try:
        response = requests.get(robots_url, timeout=10,
                               headers={'User-Agent': 'Mozilla/5.0'})
        if response.status_code == 200:
            print(f"=== {site_url} robots.txt ===")
            print(response.text[:1000])
        else:
            print(f"robots.txt 없음 (상태 코드: {response.status_code})")
    except Exception as e:
        print(f"오류: {e}")

# 연습용 사이트 확인
check_robots('https://bgzt.co.kr/')


=== https://bgzt.co.kr/ robots.txt ===
User-Agent: *
Allow: /

Sitemap: https://bgzt.co.kr/sitemap.xml



In [14]:
import urllib.robotparser

def can_fetch_url(site_url, target_url, user_agent='Mozilla/5.0'):
    """
    robots.txt 기준으로 특정 URL 접근 가능 여부를 확인합니다.
    """
    robots_url = site_url.rstrip('/') + '/robots.txt'

    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    rp.read()

    return rp.can_fetch(user_agent, target_url)


site_url = 'https://www.saramin.co.kr/'

target_url = 'https://www.saramin.co.kr/zf_user/search/recruit?searchword=파이썬'

allowed = can_fetch_url(site_url, target_url)

print(f"접근 가능 여부: {allowed}")

접근 가능 여부: True


In [13]:
# 사람인 robots.txt 확인
check_robots('https://www.saramin.co.kr')


=== https://www.saramin.co.kr robots.txt ===
# robots.txt for https://www.saramin.co.kr/
User-agent: Mediapartners-Google
Disallow:

User-agent: ia_archiver
Disallow:

User-agent: AdsBot-Google
Disallow:

User-agent: AdsBot-Google-Mobile
Disallow:

User-agent: GPTBot
Disallow: /

User-agent: Bytespider
Disallow: /

User-agent: *
Disallow: /feed.php
Disallow: /zf_user/jobs/recent-contents/company
Disallow: /ad
Disallow: /zf_user/memcom/talent-pool/main
Disallow: /zf_user/memcom/applicant/manage
Allow:    /ads.txt
Disallow: /zf_user/auth
Disallow: /zf_user/mandb
Disallow: /zf_user/mandb/view
Disallow: /zf_user/talent
Disallow: /zf_user/talent/search
Allow:    /zf_user/talent/browse-resume
Disallow: /zf_user/nonmember/apply
Disallow: /zf_user/member/apply/
Disallow: /zf_user/member/resume/
Disallow: /zf_user/free-consulting-resume/
Disallow: /zf_user/help/event-intro-consulting/
Allow:    /zf_user/help/event-intro-consulting/main
Allow:    /zf_user/help/event-intro-consulting/list
Disallo

## ✅ 배운 것 정리

```python
# 크롤링 기본 템플릿 — 앞으로 항상 이 구조 사용!

import requests
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# 1단계: robots.txt 확인 (습관!)
# 2단계: 페이지 가져오기
response = requests.get('URL', headers=headers, timeout=15)
response.encoding = 'utf-8'  # 한글 깨짐 방지

# 3단계: 상태 코드 확인
if response.status_code == 200:
    print("성공!")
    # 4단계: BeautifulSoup으로 파싱 (다음 노트북에서!)
```

> **다음 노트북**: BeautifulSoup으로 원하는 데이터만 골라내기!
